# dim_restaurant_scd2 (SCD 2, poc `scd_type_2_new` method)

One row per restaurant × version. Same content as the old `dim_restaurant` (built by `gold/restaurants`, removed 2026-11-06; see git history), written the poc way.

- **Key:** `restaurant_id` (durable: `md5('place:' || place_id)`, or name + district for a restaurant only located by Photon). No surrogate key (datle 2026-11-02): facts store `restaurant_id`; the version at time T is `start_date <= T < end_date`, the current one `current_flag = 'Y'`. `restaurant_id = 'unknown'` = Unknown row.
- **Major (new version):** `address_full`, `ward_code`, `district_code`, `city_code`, `latitude`, `longitude`, `place_id` (`scd_checksum`). **Minor:** every other column, updated in place on the current version (poc behaviour; old `dim_restaurant` also rewrote closed versions).
- **Source is rebuilt in full every run**: a restaurant combines 5 Silver tables (mentions with their geocode columns, Place Details, Text Search, OSM), so there is no date window. The checksums decide what is written: an unchanged restaurant writes nothing.
- A restaurant that left the source (its id changed, e.g. Photon id → Google place id) gets its current version closed.
- `load_type = full` (or the table does not exist): rebuild from scratch, one current version per restaurant (history lost).

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
path = f"{gold}.dim_restaurant_scd2"
if not spark.catalog.tableExists(path):
    load_type = "full"
print(f"load_type = {load_type}")

# Major (SCD 2) columns: a change closes the current version and opens a new one. Everything else is minor (in place).
scd_cols = ["address_full", "ward_code", "district_code", "city_code", "latitude", "longitude", "place_id"]

# Source tables
DETAILS = f"{silver}.stg_restaurant_place_details_latest"  # Google Place Details, latest read per place (view, stg/restaurant_place_details)
PLACES = f"{silver}.stg_places_latest"  # Google Maps Text Search, latest read of open food places (view, stg/places), Google-only restaurants
OSM = f"{silver}.stg_osm_restaurants"  # OpenStreetMap food places (stg/osm_restaurants), fills cuisine / opening hours
OSM_MAX_DISTANCE_M = 100
for src_table, nb_name in [(DETAILS, "stg/restaurant_place_details")]:
    if not spark.catalog.tableExists(src_table):
        raise RuntimeError(f"{src_table} does not exist: run {nb_name} first")
if not spark.catalog.tableExists(PLACES):  # before the first Places search: no Google-only restaurant
    PLACES = """(SELECT CAST(NULL AS STRING) AS place_id, CAST(NULL AS STRING) AS place_name,
        CAST(NULL AS STRING) AS restaurant_name_normalized, CAST(NULL AS STRING) AS formatted_address,
        CAST(NULL AS STRING) AS ward, CAST(NULL AS STRING) AS district, CAST(NULL AS STRING) AS city,
        CAST(NULL AS DOUBLE) AS latitude, CAST(NULL AS DOUBLE) AS longitude, CAST(NULL AS STRING) AS primary_type,
        CAST(NULL AS ARRAY<STRING>) AS place_types, CAST(NULL AS DOUBLE) AS rating, CAST(NULL AS INT) AS rating_count,
        CAST(NULL AS STRING) AS price_level, CAST(NULL AS BIGINT) AS price_start_vnd, CAST(NULL AS BIGINT) AS price_end_vnd,
        CAST(NULL AS STRING) AS google_maps_uri, CAST(NULL AS STRING) AS website_uri,
        CAST(NULL AS STRING) AS phone_number, CAST(NULL AS STRING) AS google_opening_hours,
        CAST(NULL AS STRING) AS business_status WHERE false)"""
    print("silver.stg_places_latest does not exist yet: no Google-only restaurants")
if not spark.catalog.tableExists(OSM):  # before the first osm_ingest: no OSM match
    OSM = """(SELECT CAST(NULL AS STRING) AS osm_key, CAST(NULL AS STRING) AS restaurant_name_normalized,
        CAST(NULL AS STRING) AS cuisine, CAST(NULL AS STRING) AS opening_hours, CAST(NULL AS DOUBLE) AS latitude,
        CAST(NULL AS DOUBLE) AS longitude WHERE false)"""
    print("silver.stg_osm_restaurants does not exist yet: no OpenStreetMap match")

# Google place type -> cuisine shown in the dashboard. Order = priority when a place has several types.
CUISINE_TYPES = [
    ("Món Việt", ["vietnamese_restaurant"]),
    ("Món Hoa", ["chinese_restaurant", "cantonese_restaurant", "dim_sum_restaurant", "chinese_noodle_restaurant",
                 "taiwanese_restaurant"]),
    ("Món Nhật", ["japanese_restaurant", "sushi_restaurant", "ramen_restaurant", "japanese_curry_restaurant",
                  "yakiniku_restaurant", "izakaya_restaurant", "tonkatsu_restaurant"]),
    ("Món Hàn", ["korean_restaurant", "korean_barbecue_restaurant"]),
    ("Món Thái", ["thai_restaurant"]),
    ("Món Ấn", ["indian_restaurant", "north_indian_restaurant", "south_indian_restaurant"]),
    ("Món Âu / Mỹ", ["french_restaurant", "italian_restaurant", "pizza_restaurant", "spanish_restaurant",
                     "greek_restaurant", "mediterranean_restaurant", "german_restaurant", "british_restaurant",
                     "european_restaurant", "american_restaurant", "steak_house", "hamburger_restaurant",
                     "mexican_restaurant", "brazilian_restaurant"]),
    ("Món quốc tế khác", ["indonesian_restaurant", "malaysian_restaurant", "filipino_restaurant", "middle_eastern_restaurant",
                          "lebanese_restaurant", "turkish_restaurant", "afghani_restaurant", "african_restaurant"]),
    ("Hải sản", ["seafood_restaurant"]),
    ("Lẩu & nướng", ["hot_pot_restaurant", "barbecue_restaurant"]),
    ("Chay", ["vegan_restaurant", "vegetarian_restaurant"]),
    ("Buffet", ["buffet_restaurant"]),
    ("Ăn nhanh", ["fast_food_restaurant", "sandwich_shop", "snack_bar"]),
    ("Cafe & tráng miệng", ["cafe", "coffee_shop", "tea_house", "dessert_shop", "dessert_restaurant", "bakery",
                            "cake_shop", "ice_cream_shop", "juice_shop", "confectionery"]),
    ("Món Á khác", ["asian_restaurant"]),   # generic: only when no specific type matched
]
# Used when Google has no specific type (most Vietnamese street food is just "restaurant") or only a
# generic one (GENERIC_CUISINES): keywords in the unaccented restaurant name, first match wins
GENERIC_CUISINES = ("Chưa rõ", "Món Á khác")
CUISINE_NAME_KEYWORDS = [
    ("Buffet", r"\\bbuffet\\b"),
    ("Lẩu & nướng", r"\\b(lau|nuong|bbq)\\b"),
    ("Hải sản", r"\\b(hai san|oc)\\b"),
    ("Món Việt", r"\\b(pho|bun|com tam|com|banh mi|banh xeo|banh cuon|banh canh|hu tieu|mi quang|bo kho|chao|xoi|bo ne)\\b"),
    ("Cafe & tráng miệng", r"\\b(cafe|ca phe|coffee|tra sua|che|kem)\\b"),
]
# restaurant_type (2026-11-02): Google primary type first, then words in the unaccented name
RESTAURANT_TYPES = [
    ("Cà phê / trà", ["cafe", "coffee_shop", "tea_house", "juice_shop", "internet_cafe", "cat_cafe", "dog_cafe"]),
    ("Bar / pub", ["bar", "pub", "wine_bar", "cocktail_bar", "lounge_bar", "sports_bar", "brewpub", "beer_garden",
                   "bar_and_grill", "night_club", "karaoke"]),
    ("Tiệm bánh / tráng miệng", ["bakery", "cake_shop", "dessert_shop", "dessert_restaurant", "ice_cream_shop",
                                 "confectionery", "donut_shop", "pastry_shop", "chocolate_shop", "candy_store"]),
    ("Ăn nhanh", ["fast_food_restaurant", "sandwich_shop", "snack_bar", "hamburger_restaurant", "hot_dog_stand"]),
    ("Buffet", ["buffet_restaurant"]),
    ("Khu ẩm thực", ["food_court"]),
]
RESTAURANT_NAME_RX = r"\\b(nha hang|restaurant|bistro|steakhouse)\\b"
EATERY_NAME_RX = r"\\b(quan|xe|tiem|hem|pho|bun|com|banh|hu tieu|mi|oc|lau|chao|xoi|che|bo ne|nuong|an vat)\\b"
spark.createDataFrame(
    [(t, cuisine, prio) for prio, (cuisine, types) in enumerate(CUISINE_TYPES) for t in types],
    "place_type STRING, cuisine STRING, priority INT",
).createOrReplaceTempView("cuisine_types")

# --- GSO (TCTK) administrative units: new wards (after 01/07/2025) and old districts (before 2025) ---
# Built by scripts/build_admin_reference.py from the vietnamadminunits package (MIT, GSO data)
import os
import pandas as pd

def _ref(name):
    path = next(p for p in [f"../../reference/{name}", f"src/reference/{name}"] if os.path.exists(p))
    return pd.read_csv(path, dtype=str, keep_default_na=False)

spark.createDataFrame(_ref("vn_new_ward_to_old_district.csv")).createOrReplaceTempView("ref_ward_raw")
spark.createDataFrame(_ref("vn_old_districts.csv")).createOrReplaceTempView("ref_district_raw")
ADMIN_KEY = "regexp_replace(lower(unaccent_vi(trim({}))), '[^a-z0-9]', '')"   # 'Phường Bến Thành' -> 'phuongbenthanh'
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ref_ward AS
    SELECT province_code AS city_code, province AS city_name,
           regexp_replace(province_key, '^(thanhpho|tinh)', '') AS city_key,
           ward_code, ward AS ward_name, ward_key, ward_short_key,
           CAST(ward_lat AS DOUBLE) AS ward_lat, CAST(ward_lon AS DOUBLE) AS ward_lon,
           nullif(district_old_code, '') AS district_old_code, nullif(districts_old, '') AS districts_old
    FROM ref_ward_raw
""")
spark.sql("""   -- ward name key -> ward, only keys that name one ward in the province
    CREATE OR REPLACE TEMP VIEW ref_ward_key AS
    SELECT city_code, match_key, ward_code
    FROM (SELECT *, COUNT(*) OVER (PARTITION BY city_code, match_key) AS n
          FROM (SELECT DISTINCT w.city_code, k AS match_key, w.ward_code
                FROM ref_ward w LATERAL VIEW explode(array(w.ward_key, w.ward_short_key)) t AS k))
    WHERE n = 1
""")
spark.sql("""   -- district alias key ('quan7', 'q7', 'govap', 'q2' -> Thu Duc) -> old district, unique in the (new) province
    CREATE OR REPLACE TEMP VIEW ref_district_key AS
    SELECT city_code, district_code, district_name, match_key
    FROM (SELECT *, COUNT(*) OVER (PARTITION BY city_code, match_key) AS n
          FROM (SELECT DISTINCT d.new_province_code AS city_code, d.district_code, d.district AS district_name,
                       regexp_replace(a, '[^a-z0-9]', '') AS match_key
                FROM ref_district_raw d LATERAL VIEW explode(split(d.aliases, '[|]')) t AS a))
    WHERE n = 1
""")
print(f"GSO reference: {spark.table('ref_ward').count()} new wards, {spark.table('ref_district_raw').count()} old districts")


run_ts = spark.sql("SELECT current_timestamp() AS ts").first()["ts"]   # start_date of new versions, end_date of closed ones
print(f"run_ts = {run_ts}")

# Read Source Tables

Every mention with its durable `restaurant_id`, then one row per restaurant (latest mention wins; Google-only places from Text Search added).

In [ ]:
# --- Every mention with its durable restaurant_id (dedup rule above; NULL until geocode matched it) ---
spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW mentions_keyed AS
    -- geocode columns live in stg_restaurant_mentions since 2026-11-06 (stg/restaurant_geocode writes them;
    -- rejected candidates keep only match_status, so their place_id / latitude are NULL)
    SELECT m.*,
           -- durable id from the Google place (same place = same restaurant); a restaurant only located by
           -- Photon (osm_approx, no place_id, 2026-11-03) is keyed by name + district until a Google place
           -- matches it (then it gets the place id, the old id is closed); NULL = not located yet
           CASE WHEN m.match_status IN ('matched', 'places_match') AND m.place_id IS NOT NULL
                THEN md5('place:' || m.place_id)
                WHEN m.match_status = 'osm_approx'
                THEN md5('name:' || m.restaurant_name_normalized || '|' || coalesce(m.district_norm, '')) END AS restaurant_id
    FROM {silver}.stg_restaurant_mentions m
""")

# --- One row per restaurant_id: latest mention wins (prefer rows with an address) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY restaurant_id ORDER BY place_id IS NULL, address IS NULL, extracted_at DESC, video_id) AS rn
        FROM mentions_keyed
        WHERE restaurant_id IS NOT NULL   -- not geocoded yet: not in the dimension (fact: Unknown)
    )
    SELECT restaurant_id,
           COALESCE(place_name, restaurant_name)    AS restaurant_name,
           restaurant_name_normalized,
           COALESCE(formatted_address, address)     AS address,
           geo_ward,                                -- Google's (new) ward, parsed from the formatted address
           COALESCE(geo_city, city)                 AS city,
           district                                 AS video_district,   -- old district as said in the video
           latitude, longitude, place_id,
           true AS has_youtube_review
    FROM ranked WHERE rn = 1
    UNION ALL
    -- Google-only: places found by Text Search that no video mentions (same md5('place:' || place_id) id).
    -- Their district is the search grid's keyword, not a fact about the place: not used.
    SELECT md5('place:' || p.place_id), p.place_name, p.restaurant_name_normalized, p.formatted_address, p.ward,
           p.city, CAST(NULL AS STRING), p.latitude, p.longitude, p.place_id, false
    FROM {PLACES} p
    WHERE p.place_id IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM mentions_keyed m WHERE m.restaurant_id = md5('place:' || p.place_id))
""")

## Google Place Details (one row per restaurant_id)

In [ ]:
# --- Type 1 attributes from Google Place Details (one row per restaurant_id) ---
# Place Details columns added 2026-11-02 (stg/restaurant_place_details adds them; NULL until a place is re-read)
details_cols = set(spark.table(DETAILS).columns)
def dcol(c):
    return f"p.{c}" if c in details_cols else "CAST(NULL AS STRING)"
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_details AS
    WITH d AS (   -- Place Details when fetched, else the same Enterprise fields from the Text Search result
        SELECT s.restaurant_id, p.primary_type, p.primary_type_display_name, p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, p.photo_name,
               p.website_uri, p.phone_number, p.google_opening_hours,
               {dcol('business_status')} AS business_status, {dcol('opening_periods')} AS opening_periods,
               {dcol('google_ward')} AS google_ward, {dcol('google_district')} AS google_district,
               {dcol('google_city')} AS google_city
        FROM restaurant_src s
        JOIN {DETAILS} p ON p.place_id = s.place_id AND p.fetch_status = 'ok'
        UNION ALL
        SELECT s.restaurant_id, p.primary_type, CAST(NULL AS STRING), p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, CAST(NULL AS STRING),
               p.website_uri, p.phone_number, p.google_opening_hours,
               p.business_status, CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING)
        FROM restaurant_src s
        JOIN {PLACES} p ON p.place_id = s.place_id
        LEFT ANTI JOIN {DETAILS} x ON x.place_id = s.place_id AND x.fetch_status = 'ok'
    ),
    typed AS (   -- primary type first, then the other Google types in their order
        SELECT d.restaurant_id, t.place_type, -1 AS pos FROM d LATERAL VIEW explode(array(d.primary_type)) t AS place_type
        UNION ALL
        SELECT d.restaurant_id, t.place_type, t.pos FROM d LATERAL VIEW posexplode(d.place_types) t AS pos, place_type
    ),
    cuisine AS (   -- most specific cuisine first (CUISINE_TYPES order), then Google's type order
        SELECT restaurant_id, min_by(c.cuisine, struct(c.priority, t.pos)) AS cuisine
        FROM typed t JOIN cuisine_types c ON c.place_type = t.place_type
        GROUP BY restaurant_id
    ),
    priced AS (
        SELECT d.*, COALESCE((price_start_vnd + price_end_vnd) / 2, price_start_vnd) AS price_mid_vnd FROM d
    )
    SELECT p.restaurant_id,
           COALESCE(c.cuisine, 'Chưa rõ')                       AS google_cuisine,
           p.primary_type_display_name                          AS cuisine_detail,
           CASE
               WHEN p.price_mid_vnd <  100000 THEN '1 · Dưới 100k'
               WHEN p.price_mid_vnd <  300000 THEN '2 · 100k–300k'
               WHEN p.price_mid_vnd <  600000 THEN '3 · 300k–600k'
               WHEN p.price_mid_vnd >= 600000 THEN '4 · Trên 600k'
               WHEN p.price_level IN ('PRICE_LEVEL_FREE', 'PRICE_LEVEL_INEXPENSIVE') THEN '1 · Dưới 100k'
               WHEN p.price_level = 'PRICE_LEVEL_MODERATE'       THEN '2 · 100k–300k'
               WHEN p.price_level = 'PRICE_LEVEL_EXPENSIVE'      THEN '3 · 300k–600k'
               WHEN p.price_level = 'PRICE_LEVEL_VERY_EXPENSIVE' THEN '4 · Trên 600k'
               ELSE 'Chưa rõ'
           END                                                  AS price_band,
           p.price_start_vnd                                    AS price_min_vnd,
           p.price_end_vnd                                      AS price_max_vnd,
           p.google_maps_uri,
           p.photo_name,
           p.website_uri                                        AS google_website,
           p.phone_number                                       AS google_phone,
           p.google_opening_hours,
           p.primary_type,
           p.business_status, p.opening_periods, p.google_ward, p.google_district, p.google_city
    FROM priced p
    LEFT JOIN cuisine c ON c.restaurant_id = p.restaurant_id
""")

## OpenStreetMap match (nearest OSM place within 100 m with a similar name)

In [ ]:
# --- OpenStreetMap match: nearest OSM place within OSM_MAX_DISTANCE_M with a similar name ---
deg = OSM_MAX_DISTANCE_M / 111000 * 1.5   # bounding box (degrees) before the exact distance
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_osm AS
    WITH c AS (
        SELECT s.restaurant_id, o.osm_key, o.cuisine AS osm_cuisine, o.opening_hours,
               s.restaurant_name_normalized AS a, o.restaurant_name_normalized AS b,
               2 * 6371000 * asin(sqrt(
                   pow(sin(radians(o.latitude - s.latitude) / 2), 2)
                   + cos(radians(s.latitude)) * cos(radians(o.latitude)) * pow(sin(radians(o.longitude - s.longitude) / 2), 2)
               )) AS distance_m
        FROM restaurant_src s
        JOIN {OSM} o
          ON o.latitude BETWEEN s.latitude - {deg} AND s.latitude + {deg}
         AND o.longitude BETWEEN s.longitude - {deg} AND s.longitude + {deg}
        WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
          AND COALESCE(s.restaurant_name_normalized, '') <> '' AND COALESCE(o.restaurant_name_normalized, '') <> ''
    ),
    ok AS (
        SELECT * FROM c
        WHERE distance_m <= {OSM_MAX_DISTANCE_M}
          AND (a = b
               OR (length(b) >= 4 AND instr(a, b) > 0)
               OR (length(a) >= 4 AND instr(b, a) > 0)
               OR levenshtein(a, b) <= greatest(2, CAST(0.25 * greatest(length(a), length(b)) AS INT)))
    )
    SELECT restaurant_id,
           min_by(osm_key, struct(distance_m, osm_key))       AS osm_key,
           min_by(osm_cuisine, struct(distance_m, osm_key))   AS osm_cuisine,
           min_by(opening_hours, struct(distance_m, osm_key)) AS opening_hours
    FROM ok GROUP BY restaurant_id
""")

## What the reviews say: dishes, prices, opening hours

In [ ]:
# --- What the reviews say (stg_restaurant_mentions, EXTRACTION_VERSION 2): dishes, prices, opening hours ---
has_dishes = "dishes" in spark.table(f"{silver}.stg_restaurant_mentions").columns
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_video_menu AS
    WITH d AS (
        SELECT m.restaurant_id, trim(x.name) AS dish, lower(unaccent_vi(trim(x.name))) AS dish_key,
               CASE WHEN x.price_vnd BETWEEN 5000 AND 5000000 THEN x.price_vnd END AS price_vnd,
               coalesce(x.is_signature, false) AS is_signature, m.video_id
        FROM mentions_keyed m
        LATERAL VIEW explode({'m.dishes' if has_dishes else 'CAST(NULL AS ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>)'}) t AS x
        WHERE m.restaurant_id IS NOT NULL AND length(trim(x.name)) > 1
    ),
    per_dish AS (
        SELECT restaurant_id, dish_key, max_by(dish, length(dish)) AS dish,
               COUNT(DISTINCT video_id) AS videos, COUNT_IF(is_signature) AS signature_votes
        FROM d GROUP BY restaurant_id, dish_key
    ),
    menu AS (
        SELECT restaurant_id,
               array_join(slice(transform(filter(array_sort(collect_list(struct(-signature_votes AS s, -videos AS v, dish))),
                                                 x -> x.s < 0), x -> x.dish), 1, 5), ', ') AS signature_dishes,
               array_join(slice(transform(array_sort(collect_list(struct(-videos AS v, -signature_votes AS s, dish))),
                                          x -> x.dish), 1, 10), ', ') AS menu_dishes
        FROM per_dish GROUP BY restaurant_id
    ),
    price AS (
        SELECT restaurant_id, MIN(price_vnd) AS video_price_min_vnd, MAX(price_vnd) AS video_price_max_vnd,
               percentile_approx(price_vnd, 0.5) AS video_price_median_vnd
        FROM d WHERE price_vnd IS NOT NULL GROUP BY restaurant_id
    ),
    hours AS (
        SELECT restaurant_id, max_by({'opening_hours' if has_dishes else 'CAST(NULL AS STRING)'}, extracted_at) AS video_opening_hours
        FROM mentions_keyed
        WHERE restaurant_id IS NOT NULL AND {'opening_hours IS NOT NULL' if has_dishes else 'false'}
        GROUP BY restaurant_id
    )
    SELECT k.restaurant_id, nullif(m.signature_dishes, '') AS signature_dishes, nullif(m.menu_dishes, '') AS menu_dishes,
           h.video_opening_hours, p.video_price_min_vnd, p.video_price_max_vnd, p.video_price_median_vnd
    FROM (SELECT DISTINCT restaurant_id FROM mentions_keyed WHERE restaurant_id IS NOT NULL) k
    LEFT JOIN menu m  ON m.restaurant_id = k.restaurant_id
    LEFT JOIN price p ON p.restaurant_id = k.restaurant_id
    LEFT JOIN hours h ON h.restaurant_id = k.restaurant_id
""")

## Address with GSO codes: province → new ward → old district

In [ ]:
# --- Address with GSO codes: province -> new ward -> old district (rules in the header) ---
K = ADMIN_KEY.format
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_location AS
    WITH s AS (
        SELECT s.restaurant_id, s.address, s.latitude, s.longitude, s.place_id, s.video_district,
               COALESCE(d.google_ward, s.geo_ward)  AS g_ward,
               d.google_district                    AS g_district,
               COALESCE(d.google_city, s.city)      AS g_city
        FROM restaurant_src s LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
    ),
    city AS (      -- province from Google's city name ('Hồ Chí Minh', 'Thành phố Hà Nội')
        SELECT s.*, c.city_code
        FROM s LEFT JOIN (SELECT DISTINCT city_code, city_key FROM ref_ward) c
          ON c.city_key = regexp_replace({K('s.g_city')}, '^(thanhpho|tinh|tp)', '')
    ),
    by_name AS (   -- new ward from Google's ward name, inside the province
        SELECT c.*, k.ward_code AS named_ward_code
        FROM city c LEFT JOIN ref_ward_key k ON k.city_code = c.city_code AND k.match_key = {K('c.g_ward')}
    ),
    nearest AS (   -- else the nearest new-ward centroid (in the province when known)
        SELECT b.restaurant_id,
               min_by(w.ward_code, pow(w.ward_lat - b.latitude, 2)
                                   + pow((w.ward_lon - b.longitude) * cos(radians(b.latitude)), 2)) AS ward_code
        FROM by_name b JOIN ref_ward w
          ON (b.city_code IS NULL OR w.city_code = b.city_code)
         AND w.ward_lat BETWEEN b.latitude - 0.5 AND b.latitude + 0.5
         AND w.ward_lon BETWEEN b.longitude - 0.5 AND b.longitude + 0.5
        WHERE b.named_ward_code IS NULL AND b.latitude IS NOT NULL AND b.longitude IS NOT NULL
        GROUP BY b.restaurant_id
    ),
    ward AS (
        SELECT b.*, COALESCE(b.named_ward_code, n.ward_code) AS ward_code,
               CASE WHEN b.named_ward_code IS NOT NULL THEN 'google' WHEN n.ward_code IS NOT NULL THEN 'nearest' END AS ward_source
        FROM by_name b LEFT JOIN nearest n ON n.restaurant_id = b.restaurant_id
    ),
    w AS (
        SELECT x.*, r.ward_name, r.district_old_code, r.districts_old,
               COALESCE(x.city_code, r.city_code) AS city_code_final, r.city_name AS ward_city_name
        FROM ward x LEFT JOIN ref_ward r ON r.ward_code = x.ward_code
    ),
    district AS (  -- old district: Google addressComponents > video > the ward's only old district
        SELECT w.*,
               gd.district_code AS google_district_code,
               COALESCE(vd.district_code, vd2.district_code) AS video_district_code
        FROM w
        LEFT JOIN ref_district_key gd ON gd.city_code = w.city_code_final AND gd.match_key = {K('w.g_district')}
        LEFT JOIN ref_district_key vd ON vd.city_code = w.city_code_final
             AND vd.match_key = regexp_replace({K('w.video_district')}, '(?<=[a-z])0+(?=[0-9])', '')
        LEFT JOIN ref_district_key vd2 ON vd2.city_code = w.city_code_final
             AND vd2.match_key = regexp_replace(regexp_replace({K('w.video_district')}, '(?<=[a-z])0+(?=[0-9])', ''),
                                                '^(tp|thanhpho|tx|thixa|huyen)', '')
    ),
    final AS (
        SELECT d.*,
               COALESCE(d.google_district_code, d.video_district_code, d.district_old_code) AS district_code,
               CASE WHEN d.google_district_code IS NOT NULL THEN 'google'
                    WHEN d.video_district_code IS NOT NULL THEN 'video'
                    WHEN d.district_old_code IS NOT NULL THEN 'ward' END AS district_source
        FROM district d
    )
    SELECT f.restaurant_id,
           f.address                                                AS address_full,
           -- street = the part before ", <ward>, ..." (else everything but the last two parts)
           nullif(trim(CASE
               WHEN f.g_ward IS NOT NULL AND f.address RLIKE concat(',\\\\s*\\\\Q', f.g_ward, '\\\\E\\\\s*(,|$)')
                   THEN regexp_replace(f.address, concat(',\\\\s*\\\\Q', f.g_ward, '\\\\E\\\\s*(,.*)?$'), '')
               WHEN size(split(f.address, ',')) >= 3
                   THEN array_join(slice(split(f.address, ','), 1, size(split(f.address, ',')) - 2), ',')
           END), '')                                                AS street,
           f.ward_code, f.ward_name, f.ward_source,
           f.district_code, rd.district_name, f.district_source, f.districts_old,
           f.city_code_final                                        AS city_code,
           COALESCE(f.ward_city_name, rc.city_name)                 AS city_name,
           f.latitude, f.longitude, f.place_id
    FROM final f
    LEFT JOIN (SELECT DISTINCT district_code, district_name FROM ref_district_key) rd ON rd.district_code = f.district_code
    LEFT JOIN (SELECT DISTINCT city_code, city_name FROM ref_ward) rc ON rc.city_code = f.city_code_final
""")
loc = spark.sql("""
    SELECT COUNT(*) AS n, COUNT(city_code) AS city, COUNT(ward_code) AS ward, COUNT_IF(ward_source = 'nearest') AS ward_nearest,
           COUNT(district_code) AS district, COUNT_IF(district_source = 'google') AS d_google,
           COUNT_IF(district_source = 'video') AS d_video, COUNT_IF(district_source = 'ward') AS d_ward
    FROM restaurant_location""").collect()[0]
print(f"Address codes: {loc['n']} restaurants · province {loc['city']} · new ward {loc['ward']} "
      f"({loc['ward_nearest']} by nearest centroid) · old district {loc['district']} "
      f"(google {loc['d_google']}, video {loc['d_video']}, ward {loc['d_ward']})")

## Opening hours: Google periods > video > OpenStreetMap

In [ ]:
# --- Opening hours: Google periods > hours said in the video > OpenStreetMap ---
PERIODS = "ARRAY<STRUCT<open: STRUCT<day: INT, hour: INT, minute: INT>, close: STRUCT<day: INT, hour: INT, minute: INT>>>"
def hhmm(m):
    return f"concat(lpad(CAST(floor(({m}) / 60) % 24 AS STRING), 2, '0'), ':', lpad(CAST(({m}) % 60 AS STRING), 2, '0'))"
# free text ('6h - 22h', '06:00-22:00', 'Mo-Su 07:00-21:30', '6h30 đến 10h tối'): first time = open, last = close
TIME_RX = "([0-9]{1,2})\\\\s*(?:h|g|giờ|:)\\\\s*([0-9]{2})?"
def text_minutes(col, idx):
    return f"""CASE WHEN size(regexp_extract_all({col}, '{TIME_RX}', 1)) >= 2 THEN
        CAST(element_at(regexp_extract_all({col}, '{TIME_RX}', 1), {idx}) AS INT) * 60
        + COALESCE(TRY_CAST(element_at(regexp_extract_all({col}, '{TIME_RX}', 2), {idx}) AS INT), 0) END"""
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_hours_text AS
    SELECT restaurant_id, src, txt,
           {text_minutes('txt', 1)} AS open_m,
           {text_minutes('txt', -1)} AS close_m0,
           txt RLIKE '(?i)(24\\\\s*/\\\\s*24|24/7|24h|cả ngày|suốt ngày|00:00\\\\s*-\\\\s*24:00)' AS is_24h
    FROM (SELECT restaurant_id, 'video' AS src, video_opening_hours AS txt FROM restaurant_video_menu WHERE video_opening_hours IS NOT NULL
          UNION ALL
          SELECT restaurant_id, 'osm', opening_hours FROM restaurant_osm WHERE opening_hours IS NOT NULL)
""")
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_hours AS
    WITH g AS (
        SELECT restaurant_id, google_opening_hours, from_json(opening_periods, '{PERIODS}') AS p
        FROM restaurant_details
    ),
    gm AS (
        SELECT restaurant_id, google_opening_hours,
               size(p) = 1 AND p[0].close IS NULL AS is_24h,   -- Google's "open 24/7": one period, no close
               array_min(transform(p, x -> x.open.hour * 60 + x.open.minute)) AS open_m,
               array_max(transform(filter(p, x -> x.close IS NOT NULL),
                                   x -> ((x.close.day - x.open.day + 7) % 7) * 1440 + x.close.hour * 60 + x.close.minute)) AS close_m
        FROM g WHERE p IS NOT NULL AND size(p) > 0
    ),
    t AS (   -- '10h tối' / '9pm': an evening closing below 12h means + 12h
        SELECT restaurant_id, src, txt, is_24h, open_m,
               CASE WHEN close_m0 < 720 AND (close_m0 <= open_m OR txt RLIKE '(?i)(tối|chiều|đêm|pm)') THEN close_m0 + 720
                    ELSE close_m0 END AS close_m
        FROM restaurant_hours_text
        WHERE is_24h OR (open_m < 1440 AND close_m0 <= 1440)
    ),
    tv AS (SELECT * FROM t WHERE src = 'video'),
    tos AS (SELECT * FROM t WHERE src = 'osm')
    SELECT s.restaurant_id,
           CASE WHEN gm.restaurant_id IS NOT NULL THEN 'google' WHEN tv.restaurant_id IS NOT NULL THEN 'video'
                WHEN tos.restaurant_id IS NOT NULL THEN 'osm' END                                     AS hours_source,
           COALESCE(gm.is_24h, tv.is_24h, tos.is_24h)                                                 AS is_open_24h,
           CASE WHEN COALESCE(gm.is_24h, tv.is_24h, tos.is_24h) THEN '00:00'
                ELSE {hhmm('COALESCE(gm.open_m, tv.open_m, tos.open_m)')} END                          AS open_time,
           CASE WHEN COALESCE(gm.is_24h, tv.is_24h, tos.is_24h) THEN '24:00'
                ELSE {hhmm('COALESCE(gm.close_m, tv.close_m, tos.close_m)')} END                       AS close_time,
           COALESCE(d.google_opening_hours, vm.video_opening_hours, o.opening_hours)                AS opening_hours_text
    FROM restaurant_src s
    LEFT JOIN gm ON gm.restaurant_id = s.restaurant_id
    LEFT JOIN tv ON tv.restaurant_id = s.restaurant_id
    LEFT JOIN tos ON tos.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_video_menu vm ON vm.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_osm o ON o.restaurant_id = s.restaurant_id
""")

## Type, status, cuisine, price

In [ ]:
# --- Final cuisine: Google specific type > keyword in the name > OSM cuisine > Google generic > Chưa rõ ---
name_rule = " ".join(f"WHEN lower(unaccent_vi(s.restaurant_name)) RLIKE '{rx}' THEN '{cuisine}'"
                     for cuisine, rx in CUISINE_NAME_KEYWORDS)
type_rule = " ".join(f"WHEN d.primary_type IN ({', '.join(repr(t) for t in types)}) THEN '{label}'"
                     for label, types in RESTAURANT_TYPES)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_attributes AS
    WITH a AS (
        SELECT s.restaurant_id, s.restaurant_name, s.has_youtube_review,
               d.google_cuisine, d.cuisine_detail, d.price_band, d.price_min_vnd, d.price_max_vnd,
               d.google_maps_uri, d.photo_name,
               d.google_website, d.google_phone, d.google_opening_hours,
               CASE {name_rule} END AS name_cuisine,
               o.osm_cuisine, o.osm_key,
               vm.signature_dishes, vm.menu_dishes,
               vm.video_price_min_vnd, vm.video_price_max_vnd, vm.video_price_median_vnd,
               CASE {type_rule} END AS google_type, d.primary_type,
               lower(unaccent_vi(s.restaurant_name)) AS name_plain,
               d.business_status,
               h.open_time, h.close_time, h.is_open_24h, h.opening_hours_text, h.hours_source
        FROM restaurant_src s
        LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_osm o ON o.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_video_menu vm ON vm.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_hours h ON h.restaurant_id = s.restaurant_id
    ),
    b AS (
        SELECT a.*,
           -- Google price first; else the median dish price stated in the reviews (2026-11-02)
           COALESCE(price_band,
                    CASE WHEN video_price_median_vnd <  100000 THEN '1 · Dưới 100k'
                         WHEN video_price_median_vnd <  300000 THEN '2 · 100k–300k'
                         WHEN video_price_median_vnd <  600000 THEN '3 · 300k–600k'
                         WHEN video_price_median_vnd >= 600000 THEN '4 · Trên 600k' END,
                    'Chưa rõ') AS final_price_band
        FROM a
    )
    SELECT restaurant_id, restaurant_name,
           CASE WHEN google_type IS NOT NULL THEN google_type
                WHEN name_plain RLIKE '{RESTAURANT_NAME_RX}' THEN 'Nhà hàng'
                WHEN name_plain RLIKE '{EATERY_NAME_RX}' THEN 'Quán ăn'
                WHEN primary_type LIKE '%restaurant%' AND left(final_price_band, 1) IN ('3', '4') THEN 'Nhà hàng'
                WHEN primary_type LIKE '%restaurant%' OR primary_type IN ('meal_takeaway', 'meal_delivery', 'food') THEN 'Quán ăn'
                ELSE 'Chưa rõ' END AS restaurant_type,
           business_status,
           NOT COALESCE(business_status = 'CLOSED_PERMANENTLY', false) AS is_active,
           open_time, close_time, is_open_24h, opening_hours_text, hours_source,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN google_cuisine
                WHEN name_cuisine IS NOT NULL THEN name_cuisine
                WHEN osm_cuisine IS NOT NULL THEN osm_cuisine
                ELSE COALESCE(google_cuisine, 'Chưa rõ') END AS cuisine,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN 'google'
                WHEN name_cuisine IS NOT NULL THEN 'name'
                WHEN osm_cuisine IS NOT NULL THEN 'osm'
                WHEN COALESCE(google_cuisine, 'Chưa rõ') <> 'Chưa rõ' THEN 'google' END AS cuisine_source,
           cuisine_detail,
           final_price_band AS price_band,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN 'google'
                WHEN video_price_median_vnd IS NOT NULL THEN 'video' END AS price_source,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN price_min_vnd ELSE video_price_min_vnd END AS price_min_vnd,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN price_max_vnd ELSE video_price_max_vnd END AS price_max_vnd,
           google_maps_uri, photo_name, has_youtube_review,
           osm_key, google_website, google_phone,
           signature_dishes, menu_dishes, video_price_min_vnd, video_price_max_vnd
    FROM b
""")
print(f"OpenStreetMap matches: {spark.table('restaurant_osm').count()}")
print(f"Restaurants in source: {spark.table('restaurant_src').count()} "
      f"(Google-only: {spark.table('restaurant_src').where('NOT has_youtube_review').count()}) · "
      f"with Google details: {spark.table('restaurant_details').count()}")

# Transformation

In [ ]:
result_df = (
    spark.table("restaurant_src").alias("s")
    .join(spark.table("restaurant_location").alias("l"), "restaurant_id")
    .join(spark.table("restaurant_attributes").alias("a"), "restaurant_id", "left")
    .select(
    F.col("restaurant_id").cast("string").alias("restaurant_id"),
    F.col("s.restaurant_name").cast("string").alias("restaurant_name"),
    F.col("s.restaurant_name_normalized").cast("string").alias("restaurant_name_normalized"),
    F.col("a.restaurant_type").cast("string").alias("restaurant_type"),
    F.col("l.address_full").cast("string").alias("address_full"),
    F.col("l.street").cast("string").alias("street"),
    F.col("l.ward_code").cast("string").alias("ward_code"),
    F.col("l.ward_name").cast("string").alias("ward_name"),
    F.col("l.ward_source").cast("string").alias("ward_source"),
    F.col("l.district_code").cast("string").alias("district_code"),
    F.col("l.district_name").cast("string").alias("district_name"),
    F.col("l.district_source").cast("string").alias("district_source"),
    F.col("l.districts_old").cast("string").alias("districts_old"),
    F.col("l.city_code").cast("string").alias("city_code"),
    F.col("l.city_name").cast("string").alias("city_name"),
    F.col("l.latitude").cast("double").alias("latitude"),
    F.col("l.longitude").cast("double").alias("longitude"),
    F.col("l.place_id").cast("string").alias("place_id"),
    F.col("a.business_status").cast("string").alias("business_status"),
    F.col("a.is_active").cast("boolean").alias("is_active"),
    F.col("a.open_time").cast("string").alias("open_time"),
    F.col("a.close_time").cast("string").alias("close_time"),
    F.col("a.is_open_24h").cast("boolean").alias("is_open_24h"),
    F.col("a.opening_hours_text").cast("string").alias("opening_hours_text"),
    F.col("a.hours_source").cast("string").alias("hours_source"),
    F.col("a.cuisine").cast("string").alias("cuisine"),
    F.col("a.cuisine_detail").cast("string").alias("cuisine_detail"),
    F.col("a.cuisine_source").cast("string").alias("cuisine_source"),
    F.col("a.price_band").cast("string").alias("price_band"),
    F.col("a.price_min_vnd").cast("bigint").alias("price_min_vnd"),
    F.col("a.price_max_vnd").cast("bigint").alias("price_max_vnd"),
    F.col("a.price_source").cast("string").alias("price_source"),
    F.col("a.google_maps_uri").cast("string").alias("google_maps_uri"),
    F.col("a.photo_name").cast("string").alias("photo_name"),
    F.col("a.google_website").cast("string").alias("google_website"),
    F.col("a.google_phone").cast("string").alias("google_phone"),
    F.col("a.osm_key").cast("string").alias("osm_key"),
    F.col("a.has_youtube_review").cast("boolean").alias("has_youtube_review"),
    F.col("a.signature_dishes").cast("string").alias("signature_dishes"),
    F.col("a.menu_dishes").cast("string").alias("menu_dishes"),
    F.col("a.video_price_min_vnd").cast("bigint").alias("video_price_min_vnd"),
    F.col("a.video_price_max_vnd").cast("bigint").alias("video_price_max_vnd"),
    F.lit(run_ts).cast("timestamp").alias("created_at"),
    F.lit(run_ts).cast("timestamp").alias("modified_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["restaurant_id"]
result_df = add_scd_checksum(result_df, scd_cols)
result_df = add_checksums(result_df, primary_key, exclude=["created_at", "modified_at"])

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        restaurant_id STRING, restaurant_name STRING, restaurant_name_normalized STRING, restaurant_type STRING,
        address_full STRING, street STRING, ward_code STRING, ward_name STRING, ward_source STRING,
        district_code STRING, district_name STRING, district_source STRING, districts_old STRING,
        city_code STRING, city_name STRING, latitude DOUBLE, longitude DOUBLE, place_id STRING,
        business_status STRING, is_active BOOLEAN, open_time STRING, close_time STRING, is_open_24h BOOLEAN,
        opening_hours_text STRING, hours_source STRING, cuisine STRING, cuisine_detail STRING, cuisine_source STRING,
        price_band STRING, price_min_vnd BIGINT, price_max_vnd BIGINT, price_source STRING,
        google_maps_uri STRING, photo_name STRING, google_website STRING, google_phone STRING, osm_key STRING,
        has_youtube_review BOOLEAN, signature_dishes STRING, menu_dishes STRING,
        video_price_min_vnd BIGINT, video_price_max_vnd BIGINT,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING, scd_checksum STRING,
        current_flag STRING, start_date TIMESTAMP, end_date TIMESTAMP
    ) USING DELTA
    COMMENT 'Restaurant dimension, SCD 2 (poc scd_type_2_new method). Key: restaurant_id + start_date (no surrogate key); restaurant_id unknown = Unknown row.'
""")

# Write to Table

`full`: overwrite, one current version per restaurant. `incremental`: `scd_type_2` (helper `_scd`): minor → update the current row; major → close it + insert a new version; new restaurant → insert.

In [ ]:
if load_type == "full":
    full_refresh(result_df.withColumn("current_flag", F.lit("Y"))
                          .withColumn("start_date", F.col("created_at"))
                          .withColumn("end_date", F.lit(MAX_DATE).cast("timestamp")), path)
    print(f"full: {spark.table(path).count()} restaurants")
else:
    counts = scd_type_2(result_df, path, keep=())
    print(f"minor (update in place): {counts['minor']} · major (new version): {counts['major']} · new restaurants: {counts['new']}")

## Close restaurants that left the source

Current versions whose `restaurant_id` is no longer built (a Photon-located restaurant that a Google place later matched gets the place id). Staged DataFrame + MERGE, because Delta rejects subqueries in MERGE conditions.

In [ ]:
to_close = (spark.table(path).filter("current_flag = 'Y' AND restaurant_id <> 'unknown'")
            .select("primary_checksum")
            .join(result_df.select("primary_checksum"), "primary_checksum", "left_anti"))
print(f"closed (left the source): {to_close.count()}")
(DeltaTable.forName(spark, path).alias("t")
    .merge(to_close.alias("s"), "t.primary_checksum = s.primary_checksum AND t.current_flag = 'Y'")
    .whenMatchedUpdate(set={"current_flag": "'N'", "end_date": f"TIMESTAMP'{run_ts}'", "etl_date": "current_timestamp()"})
    .execute())

## Unknown row

In [ ]:
if spark.table(path).where("restaurant_id = 'unknown'").isEmpty():
    spark.sql(f"""
        INSERT INTO {path} (restaurant_id, restaurant_name, restaurant_type, is_active, etl_date, etl_by,
                            current_flag, start_date, end_date)
        VALUES ('unknown', 'Unknown', 'Chưa rõ', true, current_timestamp(), 'Databricks',
                'Y', TIMESTAMP '1900-01-01', TIMESTAMP '{MAX_DATE}')
    """)

## Check: one current version per restaurant, no start_date > end_date

In [ ]:
c = spark.sql(f"""
    SELECT COUNT(*) AS n,
           COUNT_IF(current_flag = 'Y') AS cur,
           COUNT(DISTINCT CASE WHEN current_flag = 'Y' THEN restaurant_id END) AS cur_ids,
           COUNT_IF(start_date > end_date) AS bad_dates
    FROM {path} WHERE restaurant_id <> 'unknown'
""").first()
assert c["cur"] == c["cur_ids"], f"{path}: a restaurant has more than one current version"
assert c["bad_dates"] == 0, f"{path}: {c['bad_dates']} rows with start_date > end_date"
print(f"✅ {path}: {c['n']} versions, {c['cur']} current restaurants")
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").first()
print(last["operation"], dict(last["operationMetrics"] or {}))